# 02 — Forecasting the intraday volatility curves

Thesis Chapter 6. One-step-ahead forecast of the 2026-01-02 intraday curves from the FFM estimated in notebook `01`:
forecast the scalar factor score $\hat u_{1,T+1}$ (random walk, AR(p)), forecast each idiosyncratic curve
$\hat\xi_{i,T+1}(\tau)$ (random walk, FAR, HAR), then recombine $\hat x = \hat\mu_i + \sigma(\hat\chi + \hat\xi)$ on the original scale.

Requires the shared objects saved by `01_data_eda_ffm_rolling.ipynb` (`data/shared/`).

## Contents

1. [Imports & shared objects](#Imports-&-shared-objects)
2. [Factor score: stationarity and dependence](#Factor-score:-stationarity-and-dependence) — ADF, KPSS, ACF/PACF
3. [Forecasting the factor score](#Forecasting-the-factor-score) — random walk, AR(p)
4. [Idiosyncratic component](#Forecasting-Idiosyncratic-component) — functional Ljung-Box test, RW, FAR, HAR
5. [Curve reconstruction](#Forecast-total)
6. [Comparison of all model combinations](#Comparison-of-all-model-combinations) — aggregate and per pair
7. [Loading instability and forecast degradation](#Robustness-—-degradation-of-a-frozen-loading-direction)

# Import

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
from functools import partial

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf

In [ ]:
# ── Data processing & factor model ───────────────────────────────────────────

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────

In [ ]:
# ── Forecasting ───────────────────────────────────────────────────────────────
from ffm_forecasting import (
    test_stationarity,
    plot_acf_pacf,
    test_ljung_box_func,
    forecast_ar,
    plot_forecast_idiosyncratic,
    reconstruct_forecast,
    plot_forecast_vs_observed,
    select_p_far,
    forecast_idio_har,
    forecast_factor_rw,
    forecast_idio_rw,
    run_model_comparison,
    plot_model_comparison,
    run_model_comparison_by_pair,
    plot_model_comparison_by_pair,
    forecast_all_assets_FAR,
    plot_idio_forecast_comparison,
    plot_reconstructed_forecast,
)

In [ ]:
# ── Stability ─────────────────────────────────────────────────────────────────

In [ ]:
%load_ext autoreload
%autoreload 2

# Imports & shared objects

Load the objects computed and saved by `01_data_eda_ffm_rolling.ipynb`.

In [ ]:
# ── Load shared objects (saved by 01_data_eda_ffm_rolling) ────────────────────
import joblib

shared_dir = "../data/shared/"

ffm               = joblib.load(f"{shared_dir}ffm.pkl")
vol_ffm_all       = joblib.load(f"{shared_dir}vol_ffm_all.pkl")
vol_matrices_all  = joblib.load(f"{shared_dir}vol_matrices_all.pkl")
vol_matrices_2026 = joblib.load(f"{shared_dir}vol_matrices_2026.pkl")
vol_ffm_2026      = joblib.load(f"{shared_dir}vol_ffm_2026.pkl")
avg_vol           = joblib.load(f"{shared_dir}avg_vol.pkl")
FX_all            = joblib.load(f"{shared_dir}FX_all.pkl")

U_hat      = ffm['U_hat']
pairs_name = ffm['pairs']
Xi_hat     = ffm['Xi_hat']

print(f"ffm loaded  — T={ffm['U_hat'].shape[1]} days, N={len(ffm['pairs'])} pairs")
print(f"vol_ffm_all — {len(vol_ffm_all)} pairs")

# Factor score: stationarity and dependence

## ADF & KPSS

In [ ]:
U_hat = ffm['U_hat']
test_stationarity(U_hat.T, 40)

## ACF & PACF

In [ ]:
plot_acf_pacf(U_hat.T)

# Forecasting the factor score

## Random Walk

In [ ]:
fc_rw, ci_l_rw, ci_u_rw = forecast_factor_rw(
    series  = ffm['U_hat'][0, :],
    horizon = [1, 2, 3, 4, 5],
    n_display = 50, 
    plot = True
)


## AR(p)

In [ ]:
result_ar, fc_ar, ci_l_ar, ci_u_ar = forecast_ar(U_hat.T, h=5, p_max= 20,n_display=50, plot= True)

In [ ]:
print(fc_ar)

# Forecasting Idiosyncratic component

### Test Ljung-Box functional 
Idiosyncratic component white noise ?

if test rejected, not white noise - FAR(1)

In [ ]:
pairs_name = ffm['pairs']
Xi_hat      = ffm['Xi_hat']

rejected =  test_ljung_box_func(Xi_hat, pairs_name,lags = [20,50], n_pc =5)

p-values = 0 for every pair $\Rightarrow$ the factor does not capture all the common dynamics

$\Rightarrow$ the idiosyncratic components still contain a lot of structure. 

In [ ]:
orders = select_p_far(
    Xi_hat     = ffm['Xi_hat'],   # (N, T, n_tau) — idiosyncratic components
    pairs_name = ffm['pairs'],    # list of pairs
    p_max      = 10,              # maximum order to test
    n_pc       = 5,               # number of PCs used for the BIC
)

print("\nSelected FAR orders:")
for pair, info in orders.items():
    print(f"  {pair:<12} → FAR({info['far_order']})   (p per PC: {info['p_per_cp']})")


### Random walk

In [ ]:
pred_rw, ci_l_rw, ci_h_rw = forecast_idio_rw(ffm['Xi_hat'], h=1)
# carry the last observation forward
plot_forecast_idiosyncratic(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    forecasts  = pred_rw,
    title      = 'Idiosyncratic — Random Walk  h=1',
)

### FAR (FPCA + VAR)

In [ ]:
fc_far = forecast_all_assets_FAR(Xi_hat, ffm['pairs'])

plot_forecast_idiosyncratic(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    forecasts  = fc_far,
    title      = 'Idiosyncratic — FAR (FPCA + VAR)  h=1',
)

### HAR

In [ ]:
pred_idio_har, ci_low_har, ci_high_har = forecast_idio_har(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    K          = 5,
    center_window = 252, 
    fit_window = 10
)

X_log_har, X_abs_har = reconstruct_forecast(fc_ar, pred_idio_har, ffm)

In [ ]:
h=1
Xi_hat = ffm['Xi_hat']
pred_idio = plot_forecast_idiosyncratic(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    forecasts  = pred_idio_har,
    title      = 'Idiosyncratic — HAR  h=1',
)

In [ ]:
from ffm_forecasting import plot_idio_forecast_comparison

plot_idio_forecast_comparison(
    ffm = ffm,   # or ffm_2025
    forecasts_dict = {
        'RW'  : (pred_rw,       ci_l_rw,     ci_h_rw),
        'FAR' : (fc_far,        None,         None),      # no CI for FAR
        'HAR' : (pred_idio_har, ci_low_har,   ci_high_har),
    },
    pairs_to_plot = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    n_last_obs    = 10,
    h             = 1,
)

# Forecast total

### Smoothing

In [ ]:
target_date = pd.Timestamp('2026-01-02')

pairs       = ffm['pairs']
tau         = ffm['tau']
N           = len(pairs)

In [ ]:
h=1
X_log, X_abs = reconstruct_forecast(fc_ar, pred_idio_har, ffm)
print(f"h={h} : {X_abs.shape}")

In [ ]:
plot_forecast_vs_observed(
    target_date       = '2026-01-02',
    ffm               = ffm,
    #X_log             = X_log,
    X_abs= X_abs,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    h                 = 1,
)


**Evaluation targets.** The reconstructed forecast is compared with

- the *raw* observed proxy (**total** error, which includes the irreducible measurement noise of the proxy), and
- the *smoothed* observed curve (**model** error, the part of the curve that is actually forecastable).

Errors are averaged over pairs and intraday points (MAFE, MSFE). With a single target date (h = 1), the comparison illustrates the specifications rather than serving as a backtest.

In [ ]:
from ffm_forecasting import plot_reconstructed_forecast

plot_reconstructed_forecast(
    ffm  = ffm,   # or ffm_2025
    models_dict = {
        'AR + HAR' : (fc_ar,pred_idio_har),
        'AR + FAR' : (fc_ar,fc_far),
        'AR + RW'  : (fc_ar,pred_rw),
    },
    target_date       = '2026-01-02',
    vol_ffm_2026      = vol_ffm_2026,
    vol_matrices_2026 = vol_matrices_2026,
    pairs_to_plot     = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    h                 = 1,
)

# Comparison of all model combinations

### Score × idiosyncratic models, 2014–2025 fit

In [ ]:
df_cmp = run_model_comparison(
    target_dates      = ['2026-01-02', '2026-01-03'],   # h=1 → jan 2, h=2 → jan 3
    ffm               = ffm,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    horizons          = [1,2],
    models_factor     = [ 'RW', 'AR'],
    models_idio       = ['RW', 'FAR', 'HAR'],
    exclude_pairs     = ['usdcnh'],
)


In [ ]:

# Export as CSV
df_cmp.to_csv('../data/results/model_comparison.csv')

# Export for comparison with the raw-data pipeline (04_raw_ffm)
joblib.dump(df_cmp, f"{shared_dir}df_cmp_smooth.pkl")


In [ ]:
plot_model_comparison(df_cmp, 'all', h=1)

#### By pair

In [ ]:
df_cmp_pair = run_model_comparison_by_pair(
    target_dates      = ['2026-01-02', '2026-01-03'],
    ffm               = ffm,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    horizons          = [1, 2],
    models_factor     = ['RW', 'AR'],
    models_idio       = ['RW', 'FAR', 'HAR'],
    exclude_pairs     = ['usdcnh'],
)

In [ ]:
plot_model_comparison_by_pair(df_cmp_pair, 'all', h=1)


In [ ]:
from ffm_forecasting import plot_model_bar_by_pair

plot_model_bar_by_pair(df_cmp_pair, period='all', h=1)

# Robustness — degradation of a frozen loading direction

For each six-month window $w$ of the rolling grid of Chapter 5, we form a one-step forecast of the common component twice: once with the leading loading direction estimated on window $w$ (the up-to-date direction), once with the direction of the January–June 2014 reference window (the frozen direction). In both cases the score is carried forward as a random walk, which Section 6.1 showed captures its one-step predictable content, and the mean curve, the scale and the idiosyncratic part are identical. The only thing that differs between the two forecasts is the orientation of the loading subspace. We record the degradation

$$\Delta_w = \mathrm{MSFE}_{\text{frozen}}(w) - \mathrm{MSFE}_{\text{fresh}}(w)$$

on the first day outside the window.

> The last cells of this section relate $\Delta_w$ to the Grassmann distances computed in `03_stability.ipynb`
> (`data/results/grass_dist_6_30_mu.parquet`): run notebook `03` first.

In [ ]:
from ffm_forecasting import (
    forecast_degradation_frozen_direction,
    plot_forecast_degradation,
)

In [ ]:
df_degradation = forecast_degradation_frozen_direction(
    vol_ffm_all,
    window_size = 6,
    step        = 30,
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
)

df_degradation.to_parquet(
    "../data/results/degradation_frozen_vs_fresh_6_30.parquet",
    engine='pyarrow',
)
df_degradation.tail()

In [ ]:
plot_forecast_degradation(df_degradation, window_size=6, step=30)

In [ ]:
print(f"Mean   Δ_w           = {df_degradation['delta'].mean():.4e}")
print(f"Median Δ_w           = {df_degradation['delta'].median():.4e}")
print(f"Share of windows with Δ_w > 0 (frozen worse) = {(df_degradation['delta'] > 0).mean():.1%}")

In [ ]:
dg_ref_6M_30d = pd.read_parquet("../data/results/grass_dist_6_30_mu.parquet", engine='pyarrow')

In [ ]:
print(dg_ref_6M_30d.head())

In [ ]:
temp1= dg_ref_6M_30d['distance_grass']
temp2= df_degradation['delta']

correlation = temp1.corr(temp2)
print(correlation)
import statsmodels.api as sm
X = sm.OLS(temp2, sm.add_constant(temp1))
results = X.fit()
print(results.summary())